# SafeBuild Inspector — training notebook

Trains **DQN, REINFORCE, PPO and A2C** (TorchRL) on the custom flood/landslide building-inspection environment,
runs **10 hyperparameter configurations per algorithm**, and builds the report figures.

Works in **Google Colab** or a local **Jupyter** notebook. Run the cells top to bottom.
After training, copy the `models/`, `logs/` and `assets/` folders back into your GitHub repo.

## 1. Get the project and install dependencies

In [ ]:
# --- Option A: clone your GitHub repo (edit the URL) ---
# !git clone https://github.com/<your-username>/<student_name>_rl_summative.git
# %cd <student_name>_rl_summative

# --- Option B: you uploaded/unzipped the project folder; just cd into it ---
# %cd /content/student_name_rl_summative

import os; print(os.getcwd())

In [ ]:
# Colab already has PyTorch; this installs TorchRL and the rest (local Jupyter: use `uv sync` instead)
!pip install -q torchrl==0.15.0 tensordict==0.15.6 "gymnasium>=1.0" pygame PyOpenGL pillow matplotlib pandas

## 2. Check the environment and see it

In [ ]:
from environment import SafeBuildInspectorEnv
from IPython.display import display
from PIL import Image

env = SafeBuildInspectorEnv(render_mode="rgb_array")
obs, info = env.reset(seed=3)
print("observation size:", obs.shape, "| actions:", env.action_space.n, "| info:", info)
for a in [1, 1, 2, 4]:
    env.step(a)
display(Image.fromarray(env.render()))

## 3. Train — 10 configurations per algorithm

Frames per run (increase for stronger agents if you have time). Rough CPU times per run:
DQN ≈ 5–8 min, PPO ≈ 6–10 min, A2C ≈ 5–8 min, REINFORCE ≈ 3–5 min.
To test the pipeline quickly first, set the frames to ~20 000.

In [ ]:
FRAMES = {"dqn": 200_000, "reinforce": 300_000, "ppo": 300_000, "a2c": 300_000}

In [ ]:
!python -m training.dqn_training --sweep --frames {FRAMES["dqn"]}

In [ ]:
!python -m training.pg_training --algo reinforce --sweep --frames {FRAMES["reinforce"]}

In [ ]:
!python -m training.pg_training --algo ppo --sweep --frames {FRAMES["ppo"]}

In [ ]:
!python -m training.pg_training --algo a2c --sweep --frames {FRAMES["a2c"]}

Tip: to run a single configuration, e.g. PPO config 3:
`!python -m training.pg_training --algo ppo --config 3 --frames 300000`

## 4. Results: tables and figures for the report

In [ ]:
!python -m training.analyze
import pandas as pd
display(pd.read_csv("assets/summary_table.csv"))
for algo in ["dqn", "reinforce", "ppo", "a2c"]:
    print(algo.upper()); display(pd.read_csv(f"assets/sweep_{algo}.csv")[
        [c for c in pd.read_csv(f"assets/sweep_{algo}.csv").columns if c not in ("algo", "run", "seed")]])

In [ ]:
for f in ["fig_cumulative_rewards", "fig_dqn_loss", "fig_pg_entropy", "fig_sweeps", "fig_generalisation"]:
    display(Image.open(f"assets/{f}.png"))

## 5. Watch the best agent (GIF, works without a display)

In [ ]:
!python main.py --render none --record --episodes 1
from IPython.display import Image as IPImage
import glob
display(IPImage(filename=sorted(glob.glob("assets/*_agent.gif"))[-1]))

## 6. (Colab) download results to put in your repo

In [ ]:
!zip -qr results.zip models logs assets
try:
    from google.colab import files; files.download("results.zip")
except ImportError:
    print("Local run: results are already in models/, logs/ and assets/")